In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from rotation.solver import initialize_system, create_func, solve_mixing
from rotation.symbolic import ANGLE, BARE, PROPAGATION, create_3d_rotation, create_mix_equation

In [ ]:
# Mass info from online PDG (in GeV)
# https://pdglive.lbl.gov/Particle.action?node=S009
# https://pdglive.lbl.gov/Particle.action?node=S014
PDG = {
    PROPAGATION["pi0"]: 0.1349768,
    PROPAGATION["eta"]: 0.547862,
}

In [ ]:
rotate = create_3d_rotation()[0]
equation = create_mix_equation(rotate)[0]
argument_order, system_of_equations = initialize_system(equation.lhs - equation.rhs)

known = {
    PROPAGATION["pi0"]: PDG[PROPAGATION["pi0"]],
    PROPAGATION["eta"]: PDG[PROPAGATION["eta"]],
    BARE["phi"]: 1.0,
    BARE["pi0-phi"]: 0.1,
    BARE["eta-phi"]: 0.3,
}

unknown = {
    PROPAGATION["phi"]: 1.0,
    BARE["pi0"]: PDG[PROPAGATION["pi0"]],
    BARE["eta"]: PDG[PROPAGATION["eta"]],
    ANGLE["alpha"]: 0.0,
    ANGLE["beta"]: 0.0,
    ANGLE["gamma"]: 0.0,
}

In [ ]:
x_label = BARE["phi"]

solutions = []
for x in np.linspace(0, known[BARE["phi"]],  num=101):
    known[x_label] = x
    func = create_func(
        system_of_equations,
        argument_order,
        known,
    )
    try:
        solved = solve_mixing(
            func,
            argument_order,
            unknown,
        )
        solutions.append([known.copy(), solved.copy()])
    except ValueError as e:
        pass

In [ ]:
xs = np.array([sol[0][x_label] for sol in solutions])
# Reorganize solutions to be arrays of each unknown
data = {}
for sol in solutions:
    for y_label, data_point in sol[1].items():
        try:
            data[y_label].append(data_point)
        except KeyError:
            data[y_label] = [data_point]
for y_label, ys in data.items():
    data[y_label] = np.array(ys)
    data[y_label] = np.ma.array(ys, mask=data[y_label] < 0.0)

if not data:
    print("No data!")
else:
    # plotting
    fig, axs = plt.subplots(1,2, figsize=(10,6))
    
    for y_label, ys in data.items():
        if y_label in ANGLE.values():
            ax = axs[1]
        else:
            ax = axs[0]
        ax.scatter(xs, ys.data, label=str(y_label), marker=".")
    axs[0].set_ylabel("Mass [GeV]")
    axs[1].set_ylabel("Angle")
    title = []
    for k, v in solutions[0][0].items():
        if k is not x_label:
            title.append(f"{k}={v:.2e}")
    fig.suptitle(", ".join(title))
    for ax in axs.flatten():
        ax.set_xlabel(x_label)
        ax.legend()
        ax.grid(True)
    fig.show()